In [1]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2t.sqlite' AS t_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2d.sqlite' AS td_db
""")

In [2]:
con.execute("""
CREATE OR REPLACE TABLE test_join AS
SELECT
    t.datetime,
    t.point_id,
    
    t."0"  AS t_0,
    t."3"  AS t_3,
    t."6"  AS t_6,
    t."9"  AS t_9,
    t."12" AS t_12,
    t."15" AS t_15,
    t."18" AS t_18,
    t."21" AS t_21,

    td."0"  AS td_0,
    td."3"  AS td_3,
    td."6"  AS td_6,
    td."9"  AS td_9,
    td."12" AS td_12,
    td."15" AS td_15,
    td."18" AS td_18,
    td."21" AS td_21

FROM t_db.daily_data t
JOIN td_db.daily_data td
ON t.datetime = td.datetime
AND t.point_id = td.point_id
LIMIT 100000
""")

In [3]:
con.execute("""
CREATE OR REPLACE TABLE test_rh_vpd_all AS
SELECT
    datetime,
    point_id,

    -- RH
    100.0 * EXP((17.625 * (td_0  - 273.15)) / (243.04 + (td_0  - 273.15))) /
    EXP((17.625 * (t_0   - 273.15)) / (243.04 + (t_0   - 273.15))) AS rh_0,

    100.0 * EXP((17.625 * (td_3  - 273.15)) / (243.04 + (td_3  - 273.15))) /
    EXP((17.625 * (t_3   - 273.15)) / (243.04 + (t_3   - 273.15))) AS rh_3,

    100.0 * EXP((17.625 * (td_6  - 273.15)) / (243.04 + (td_6  - 273.15))) /
    EXP((17.625 * (t_6   - 273.15)) / (243.04 + (t_6   - 273.15))) AS rh_6,

    100.0 * EXP((17.625 * (td_9  - 273.15)) / (243.04 + (td_9  - 273.15))) /
    EXP((17.625 * (t_9   - 273.15)) / (243.04 + (t_9   - 273.15))) AS rh_9,

    100.0 * EXP((17.625 * (td_12 - 273.15)) / (243.04 + (td_12 - 273.15))) /
    EXP((17.625 * (t_12  - 273.15)) / (243.04 + (t_12  - 273.15))) AS rh_12,

    100.0 * EXP((17.625 * (td_15 - 273.15)) / (243.04 + (td_15 - 273.15))) /
    EXP((17.625 * (t_15  - 273.15)) / (243.04 + (t_15  - 273.15))) AS rh_15,

    100.0 * EXP((17.625 * (td_18 - 273.15)) / (243.04 + (td_18 - 273.15))) /
    EXP((17.625 * (t_18  - 273.15)) / (243.04 + (t_18  - 273.15))) AS rh_18,

    100.0 * EXP((17.625 * (td_21 - 273.15)) / (243.04 + (td_21 - 273.15))) /
    EXP((17.625 * (t_21  - 273.15)) / (243.04 + (t_21  - 273.15))) AS rh_21,

    -- VPD (RAW, no clamping)
    0.6108 * EXP((17.27 * (t_0   - 273.15)) / ((t_0   - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td_0  - 273.15)) / ((td_0  - 273.15) + 237.3)) AS vpd_0,

    0.6108 * EXP((17.27 * (t_3   - 273.15)) / ((t_3   - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td_3  - 273.15)) / ((td_3  - 273.15) + 237.3)) AS vpd_3,

    0.6108 * EXP((17.27 * (t_6   - 273.15)) / ((t_6   - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td_6  - 273.15)) / ((td_6  - 273.15) + 237.3)) AS vpd_6,

    0.6108 * EXP((17.27 * (t_9   - 273.15)) / ((t_9   - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td_9  - 273.15)) / ((td_9  - 273.15) + 237.3)) AS vpd_9,

    0.6108 * EXP((17.27 * (t_12  - 273.15)) / ((t_12  - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td_12 - 273.15)) / ((td_12 - 273.15) + 237.3)) AS vpd_12,

    0.6108 * EXP((17.27 * (t_15  - 273.15)) / ((t_15  - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td_15 - 273.15)) / ((td_15 - 273.15) + 237.3)) AS vpd_15,

    0.6108 * EXP((17.27 * (t_18  - 273.15)) / ((t_18  - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td_18 - 273.15)) / ((td_18 - 273.15) + 237.3)) AS vpd_18,

    0.6108 * EXP((17.27 * (t_21  - 273.15)) / ((t_21  - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td_21 - 273.15)) / ((td_21 - 273.15) + 237.3)) AS vpd_21

FROM test_join
""")

In [4]:
con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_rh.sqlite' AS out_rh
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_vpd.sqlite' AS out_vpd
""")

In [5]:
con.execute("""
CREATE OR REPLACE TABLE daily_data AS
SELECT
    datetime,
    point_id,
    'rh' AS variable,
    rh_0  AS "0",
    rh_3  AS "3",
    rh_6  AS "6",
    rh_9  AS "9",
    rh_12 AS "12",
    rh_15 AS "15",
    rh_18 AS "18",
    rh_21 AS "21"
FROM test_rh_vpd_all
""")

In [6]:
con.execute("""
CREATE OR REPLACE TABLE daily_data AS
SELECT
    datetime,
    point_id,
    'vpd' AS variable,
    vpd_0  AS "0",
    vpd_3  AS "3",
    vpd_6  AS "6",
    vpd_9  AS "9",
    vpd_12 AS "12",
    vpd_15 AS "15",
    vpd_18 AS "18",
    vpd_21 AS "21"
FROM test_rh_vpd_all
""")

In [7]:
con.execute("""
CREATE UNIQUE INDEX idx_rh 
ON out_rh.daily_data(datetime, point_id)
""")

con.execute("""
CREATE UNIQUE INDEX idx_vpd 
ON out_vpd.daily_data(datetime, point_id)
""")

CatalogException: Catalog Error: Table with name daily_data does not exist!
Did you mean "memory.daily_data, t_db.daily_data, or td_db.daily_data"?

In [2]:
import duckdb

con = duckdb.connect()

# Load SQLite support
con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")

# Attach source DBs
con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2t.sqlite' AS t_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2d.sqlite' AS td_db
""")

# Attach output DBs
con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_rh.sqlite' AS out_rh
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_vpd.sqlite' AS out_vpd
""")

# Drop existing tables (safe reset)
con.execute("DROP TABLE IF EXISTS out_rh.daily_data")
con.execute("DROP TABLE IF EXISTS out_vpd.daily_data")

# =========================
# RH TABLE (WRITE DIRECT)
# =========================
con.execute("""
CREATE TABLE out_rh.daily_data AS
SELECT
    t.datetime,
    t.point_id,
    'rh' AS variable,

    100.0 * EXP((17.625 * (td."0"  - 273.15)) / (243.04 + (td."0"  - 273.15))) /
    EXP((17.625 * (t."0"   - 273.15)) / (243.04 + (t."0"   - 273.15))) AS "0",

    100.0 * EXP((17.625 * (td."3"  - 273.15)) / (243.04 + (td."3"  - 273.15))) /
    EXP((17.625 * (t."3"   - 273.15)) / (243.04 + (t."3"   - 273.15))) AS "3",

    100.0 * EXP((17.625 * (td."6"  - 273.15)) / (243.04 + (td."6"  - 273.15))) /
    EXP((17.625 * (t."6"   - 273.15)) / (243.04 + (t."6"   - 273.15))) AS "6",

    100.0 * EXP((17.625 * (td."9"  - 273.15)) / (243.04 + (td."9"  - 273.15))) /
    EXP((17.625 * (t."9"   - 273.15)) / (243.04 + (t."9"   - 273.15))) AS "9",

    100.0 * EXP((17.625 * (td."12" - 273.15)) / (243.04 + (td."12" - 273.15))) /
    EXP((17.625 * (t."12"  - 273.15)) / (243.04 + (t."12"  - 273.15))) AS "12",

    100.0 * EXP((17.625 * (td."15" - 273.15)) / (243.04 + (td."15" - 273.15))) /
    EXP((17.625 * (t."15"  - 273.15)) / (243.04 + (t."15"  - 273.15))) AS "15",

    100.0 * EXP((17.625 * (td."18" - 273.15)) / (243.04 + (td."18" - 273.15))) /
    EXP((17.625 * (t."18"  - 273.15)) / (243.04 + (t."18"  - 273.15))) AS "18",

    100.0 * EXP((17.625 * (td."21" - 273.15)) / (243.04 + (td."21" - 273.15))) /
    EXP((17.625 * (t."21"  - 273.15)) / (243.04 + (t."21"  - 273.15))) AS "21"

FROM t_db.daily_data t
JOIN td_db.daily_data td
ON t.datetime = td.datetime
AND t.point_id = td.point_id

-- LIMIT 100000  -- 🔁 USE FOR TESTING ONLY
""")

# =========================
# VPD TABLE (WRITE DIRECT)
# =========================
con.execute("""
CREATE TABLE out_vpd.daily_data AS
SELECT
    t.datetime,
    t.point_id,
    'vpd' AS variable,

    0.6108 * EXP((17.27 * (t."0"   - 273.15)) / ((t."0"   - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td."0"  - 273.15)) / ((td."0"  - 273.15) + 237.3)) AS "0",

    0.6108 * EXP((17.27 * (t."3"   - 273.15)) / ((t."3"   - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td."3"  - 273.15)) / ((td."3"  - 273.15) + 237.3)) AS "3",

    0.6108 * EXP((17.27 * (t."6"   - 273.15)) / ((t."6"   - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td."6"  - 273.15)) / ((td."6"  - 273.15) + 237.3)) AS "6",

    0.6108 * EXP((17.27 * (t."9"   - 273.15)) / ((t."9"   - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td."9"  - 273.15)) / ((td."9"  - 273.15) + 237.3)) AS "9",

    0.6108 * EXP((17.27 * (t."12"  - 273.15)) / ((t."12"  - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td."12" - 273.15)) / ((td."12" - 273.15) + 237.3)) AS "12",

    0.6108 * EXP((17.27 * (t."15"  - 273.15)) / ((t."15"  - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td."15" - 273.15)) / ((td."15" - 273.15) + 237.3)) AS "15",

    0.6108 * EXP((17.27 * (t."18"  - 273.15)) / ((t."18"  - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td."18" - 273.15)) / ((td."18" - 273.15) + 237.3)) AS "18",

    0.6108 * EXP((17.27 * (t."21"  - 273.15)) / ((t."21"  - 273.15) + 237.3)) -
    0.6108 * EXP((17.27 * (td."21" - 273.15)) / ((td."21" - 273.15) + 237.3)) AS "21"

FROM t_db.daily_data t
JOIN td_db.daily_data td
ON t.datetime = td.datetime
AND t.point_id = td.point_id

-- LIMIT 100000  -- 🔁 USE FOR TESTING ONLY
""")

# =========================
# INDEXES
# =========================
con.execute("""
CREATE UNIQUE INDEX idx_rh 
ON out_rh.daily_data(datetime, point_id)
""")

con.execute("""
CREATE UNIQUE INDEX idx_vpd 
ON out_vpd.daily_data(datetime, point_id)
""")

con.close()

In [6]:
con.execute("""
SELECT datetime, point_id, COUNT(*) as cnt
FROM rh_db.daily_data
GROUP BY datetime, point_id
HAVING COUNT(*) > 1
LIMIT 10
""").fetchall()

[]

In [2]:

import duckdb

con = duckdb.connect()

con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_rh.sqlite' AS rh_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_vpd.sqlite' AS vpd_db
""")


con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2t.sqlite' AS t_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2d.sqlite' AS td_db
""")

In [3]:
df = con.execute("""
SELECT *
FROM rh_db.daily_data AS rh
JOIN vpd_db.daily_data AS vpd
  ON rh.datetime = vpd.datetime
 AND rh.point_id = vpd.point_id
JOIN t_db.daily_data AS t
  ON rh.datetime = t.datetime
 AND rh.point_id = t.point_id
JOIN td_db.daily_data AS td
  ON rh.datetime = td.datetime
 AND rh.point_id = td.point_id
WHERE rh.point_id = 500
""").df()

In [5]:
df.head()

,datetime,point_id,variable,0,3,6,9,12,15,18,...,point_id_3,variable_3,0_3,3_3,6_3,9_3,12_3,15_3,18_3,21_3
0,20220101,500,rh,88.109354,86.322972,86.273041,76.201255,68.930552,70.406617,67.712648,...,500,Td,264.489502,263.540039,262.438400,259.766159,257.327087,254.914612,253.539978,253.709595
1,20220102,500,rh,73.952099,79.691699,78.863246,80.812215,81.505385,82.120548,80.144888,...,500,Td,253.260803,252.914246,251.840881,251.781555,251.562195,251.137238,251.501602,250.849014
2,20220103,500,rh,54.950787,53.672343,54.940894,57.221108,58.322442,59.063593,56.403701,...,500,Td,249.407135,249.281036,249.908493,250.432083,250.717346,250.889374,251.189972,252.767944
3,20220104,500,rh,53.396284,53.798122,57.901934,65.116091,63.816439,64.266959,58.352936,...,500,Td,251.927902,252.300903,252.690018,253.911194,254.346436,253.889069,254.467865,257.317703
4,20220105,500,rh,71.189283,65.801958,66.857079,80.208870,78.098405,70.570433,74.293829,...,500,Td,256.863785,257.640594,258.712265,260.061523,259.227997,259.625854,263.343918,267.582916


In [5]:
con.execute("PRAGMA database_list").df()

,seq,name,file
0,624,memory,None
1,2091,rh_db,/home/joe/work/Fire/ML/Data/DB/era5_daily_2982...
2,2112,vpd_db,/home/joe/work/Fire/ML/Data/DB/era5_daily_2982...
3,2115,t_db,/home/joe/work/Fire/ML/Data/DB/era5_daily_2982...
4,2119,td_db,/home/joe/work/Fire/ML/Data/DB/era5_daily_2982...


In [6]:
con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,rh_db,main,daily_data,"[datetime, point_id, variable, 0, 3, 6, 9, 12,...","[VARCHAR, BIGINT, VARCHAR, DOUBLE, DOUBLE, DOU...",False
1,t_db,main,daily_data,"[datetime, point_id, variable, 0, 3, 6, 9, 12,...","[VARCHAR, BIGINT, VARCHAR, DOUBLE, DOUBLE, DOU...",False
2,vpd_db,main,daily_data,"[datetime, point_id, variable, 0, 3, 6, 9, 12,...","[VARCHAR, BIGINT, VARCHAR, DOUBLE, DOUBLE, DOU...",False
